<a href="https://colab.research.google.com/github/Yuzhanbo-98/IJC437-python-practice/blob/main/IntroPython_part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Python and Google Colab (Part 3)

This part we will learn **pandas**, the package that all data work in Python is built on, and the **DataFrame**, load real data from a file, and learn how to clean it up.


Part 3 learning objectives:

1. To install and import packages
2. To use DataFrame to store tabular data, and select, filter and summarise it
3. To load data from a CSV file and save your results back out again
4. To find and fix the faults that real data arrives with

Exercises are marked with a pencil.


---



## 1. Recap

This section is a quick recap on Part 1 and 2.

Work through it as a quiz. Read each cell, decide what it will print, then run it and see whether you were right. Anything that surprises you is the bit to go back and look at in your Part 1 or Part 2 notebook.



### 1.1 Lists and dictionaries

A list holds things in order and is numbered from zero. A negative index counts back from the end.

In [ ]:
sports = ["archery", "boxing", "cycling", "diving"]
print(sports[0])
print(sports[-1])
print(len(sports))

archery
diving
4


A dictionary holds pairs, and you look a value up by its key rather than by position.

In [ ]:
medals_won = {"USA": 121, "GBR": 67, "CHN": 70}
print(medals_won["GBR"])
print(list(medals_won.keys()))
print(list(medals_won.values()))

67
['USA', 'GBR', 'CHN']
[121, 67, 70]


Both come back within the hour. A pandas table is built from a dictionary of lists: the keys become the column names and the lists become the columns underneath them.

### 1.2 f-strings

An f-string builds a piece of text out of the values of your variables. The `:.1%` tells Python to show that number as a percentage to one decimal place.

In [ ]:
country = "GBR"
gold = 27
total = 67
share = gold / total

print(f"{country} won {total} medals, {gold} of them gold ({share:.1%})")

GBR won 67 medals, 27 of them gold (40.3%)


### 1.3 Asking true or false questions

Filtering a table is built entirely out of comparisons like these, so they are worth a second look.

In [ ]:
print(total > 50)
print(total > 50 and gold > 30)
print("GBR" in medals_won)

True
False
True


### 1.4 Functions and lambdas

A function packages a job up under a name so you can run it whenever you like, with different values each time.

In [ ]:
def medal_points(gold, silver, bronze):
    return gold * 3 + silver * 2 + bronze

print(medal_points(27, 23, 17))

144


A lambda is the same idea on a single line, for a job too small to be worth a name. Lambdas matter today because `.apply()` takes one and runs it down a whole column of a table.

In [ ]:
to_percent = lambda x: f"{x:.1%}"

print(to_percent(27 / 67))

40.3%


### 1.5 NumPy arrays

An array does arithmetic on every value at once, with no loop in sight. A pandas column is a NumPy array underneath, so this is the habit of mind the rest of the session is built on.

In [ ]:
import numpy as np

golds = np.array([46, 27, 26, 19, 17, 12])
print(golds * 2)
print(golds.sum())
print(golds.mean())

[92 54 52 38 34 24]
147
24.5


> **Note:** keep the `print()` calls. A cell shows you only its last line, so without them you would see the mean and nothing else.

> **&#9998; Exercise.** Make a list of four cities. Print the second one and the number of cities, then add a fifth city with .append() and print the list again.

In [ ]:
# your answer here
countries = ["USA","CAN","CHN","RUS"]
print(countries[1])
print(len(countries))
countries.append("AUS")
print(countries)



CAN
4
['USA', 'CAN', 'CHN', 'RUS', 'AUS']


> **&#9998; Exercise.** Using three variables for the name, the score and the total, write an f-string that prints `Ada scored 17 out of 20 (85.0%)`.

In [ ]:
# your answer here
name = "ada"
score = 17
total = 20
print(f"{name} scored {score} out of {total}")
print(f"{name} scored {score} out of {total} ({score/total*100:.1f}%)")

ada scored 17 out of 20
ada scored 17 out of 20 (85.0%)


> **&#9998; Exercise.** Write a function that takes a number of medals and hands back the text "high" if it is above 50 and "low" otherwise. Test it on two numbers. Then write the same thing as a lambda and check the two agree.

In [ ]:
# your answer here
def medal_level (n):
 if n > 50:
  return "high"
 else:
  return "low"
print(medal_level(2))
print(medal_level(200))

low
high


## 2. Packages

A package is a bundle of ready-made functions and data that somebody else has written and shared. Almost everything Python is famous for lives in packages rather than in the language itself, so knowing how to find and load them is a skill in its own right.

Most packages live on PyPI, the Python Package Index. Colab already has several hundred of the most common ones installed, pandas and NumPy among them, so most of the time you can simply `import` what you want:

In [ ]:
import numpy as np
import pandas as pd

Run that cell now; the rest of this notebook assumes it. The `as np` part gives the package a short nickname. These two nicknames, `np` and `pd`, are near-universal conventions that you will see in every piece of Python data code you ever read.

When a package is not already installed, you fetch it from PyPI with a tool called `pip`. Inside a notebook you run pip by putting an exclamation mark in front of the line, which tells Colab “this one is not Python, run it as a system command”:

```python
!pip install packagename
```

Everything we need today is already in Colab, so you will not have to install anything. You can still see what pip does by asking it for something that is already there:

In [ ]:
!pip install pandas

The reply is `Requirement already satisfied`, which is pip telling you there is nothing to do. It is not an error, and you will meet it often.

> **Note:** anything you do install has to be installed again in every new Colab session. Google hands you a fresh machine each time, and whatever you put on the last one is gone. Keep any pip lines in a cell near the top of your notebook so they run along with everything else.

Installing and importing are two different steps. `pip install` puts the package on the machine. `import` makes it available to your code, and has to happen in every session.



## 3. DataFrame: Tables of data

At the end of Part 2 we built a medal table as a two-dimensional array and ran into its limitation: an array is all one type and is addressed by number only, so there was nowhere to put the country names.

A **DataFrame** solves exactly that. It is a table where:

* each column has a name, and holds values of a single type;
* different columns may hold different types, text in one and numbers in the next;
* each row has a label too, called the index.

If you have used a spreadsheet you already have the right mental picture. The DataFrame is the most important data structure in Python for data science, and most of what you do from here on will start with one.



### 3.1 Creating a DataFrame

The simplest way to build one is from a dictionary, where each key is a column name and each value is the list of values down that column. Here is the Rio 2016 medal table you have been building up since Part 1, this time with the countries where they belong:

In [ ]:
import pandas as pd
#IMPORT PANDAS AS PD
medals = pd.DataFrame({
    "Country": ["USA", "GBR", "CHN", "RUS", "GER", "JPN"],
    "Gold":    [46, 27, 26, 19, 17, 12],
    "Silver":  [37, 23, 18, 18, 10,  8],
    "Bronze":  [38, 17, 26, 19, 15, 21],
})
medals

,Country,Gold,Silver,Bronze
0,USA,46,37,38
1,GBR,27,23,17
2,CHN,26,18,26
3,RUS,19,18,19
4,GER,17,10,15
5,JPN,12,8,21


Each column is on its own line to make the code readable; you could type it all on one line if you preferred.

The numbers 0 to 5 down the left are the **index**, the row labels. Like everything else in Python they start at 0. They are not a column of data; they are the names of the rows.

The `dtypes` property reports the type of each column:

In [ ]:
medals.dtypes

,0
Country,object
Gold,int64
Silver,int64
Bronze,int64


(Depending on your version of pandas the Country row may read `str` rather than `object`. Both simply mean “text”, and the column spacing shifts a little to match. `int64` means a whole number.)

### 3.2 Selecting columns

Put a column name in square brackets to get that column:

In [ ]:
medals["Gold"]

,Gold
0,46
1,27
2,26
3,19
4,17
5,12


What comes back is a single column of data, which pandas calls a **Series**. You can see that from the output: it has the row labels down the left and reports its own name and type at the bottom. A Series behaves much like the NumPy arrays from Part 2, vectorised in the same way, so you can do arithmetic on a whole column at once.

There is a shorthand with a dot, which is quicker to type:

In [ ]:
medals.Gold      # the same thing

,Gold
0,46
1,27
2,26
3,19
4,17
5,12


Use it for reading a column, but it is only a shorthand: it cannot create a column, and it does not work for a name with a space in it. When in doubt, use the brackets.

If you put a *list* of column names in the brackets, you get back a smaller DataFrame rather than a single Series. Note the two sets of brackets, one for the indexing and one for the list:

In [ ]:
medals[["Country", "Gold"]]

,Country,Gold
0,USA,46
1,GBR,27
2,CHN,26
3,RUS,19
4,GER,17
5,JPN,12


> **Note:** one pair of brackets with a single name gives a Series; two pairs give a DataFrame. This catches everybody out at some point. A quick way to tell them apart is the output: a Series prints as a single list of values with a dtype at the bottom, a DataFrame prints as a table with a header row.

### 3.3 Selecting rows

Rows are selected by position with `.iloc` (short for “integer location”). It works like indexing the two-dimensional array in Part 2: rows before the comma, columns after it, and a bare colon meaning “all of them”.

In [ ]:
medals.iloc[0]       # the first row, as a Series
medals.iloc[[0]]      # the first row, as a DataFrame
medals.iloc[[0, 2]]    # the first and third rows
medals.iloc[0:3]      # the first three rows
medals.iloc[[0, 2], [0, 1]]   # first and third rows, first two columns

,Country,Gold
0,USA,46
2,CHN,26


Only the last line of a cell displays automatically, so try these one at a time to see what each returns.

In [ ]:
medals.iloc[[0, 2]]

,Country,Gold,Silver,Bronze
0,USA,46,37,38
2,CHN,26,18,26


> **&#9998; Exercise.** Select the second and fourth rows of the table. Then select those same rows but only the Country and Gold columns.

In [ ]:
# your answer here
import pandas as pd
medals.iloc[[1,3]]
medals.iloc[[1,3],[0,1]]


,Country,Gold
1,GBR,27
3,RUS,19


### 3.4 Filtering rows: asking a question of the table

This is the part you will use most. Compare a column with a value and pandas gives back a True/False answer for every row, exactly like the boolean arrays in Part 2:

In [ ]:
medals["Country"] == "GBR"

,Country
0,False
1,True
2,False
3,False
4,False
5,False


Put that inside square brackets and pandas keeps only the rows where it was True:

In [ ]:
medals[medals["Country"] == "GBR"]

,Country,Gold,Silver,Bronze
1,GBR,27,23,17


In [ ]:
medals[medals["Gold"] >= 27]

,Country,Gold,Silver,Bronze
0,USA,46,37,38
1,GBR,27,23,17


To combine two conditions, use `&` and `|`, the same symbols as for NumPy arrays in Part 2 rather than the words `and` and `or`, and give each condition its own brackets:

In [ ]:
medals[(medals["Gold"] >= 20) & (medals["Bronze"] > 20)]

,Country,Gold,Silver,Bronze
0,USA,46,37,38
2,CHN,26,18,26


When you are checking one column against several possible values, `.isin()` is tidier than a string of ors:

In [ ]:
medals[medals["Country"].isin(["CHN", "GBR"])]

,Country,Gold,Silver,Bronze
1,GBR,27,23,17
2,CHN,26,18,26


To choose rows *and* columns at once, use `.loc`, which takes a row selector and a column selector separated by a comma:

In [ ]:
medals.loc[medals["Gold"] >= 27, ["Country", "Gold"]]

,Country,Gold
0,USA,46
1,GBR,27


There is a second style that many people find easier to read, especially for longer conditions. `.query()` takes the condition as a piece of text, in which you name the columns directly with no quotes and no repetition of the DataFrame's name, and use the words `and`, `or` and `in`:

In [ ]:
medals.query("Gold >= 27")

,Country,Gold,Silver,Bronze
0,USA,46,37,38
1,GBR,27,23,17


In [ ]:
medals.query("Gold >= 20 and Bronze > 20")

,Country,Gold,Silver,Bronze
0,USA,46,37,38
2,CHN,26,18,26


Text values inside a query need their own quotes, so use single quotes inside a double-quoted query. You can also follow a query with a column selection, which picks rows and columns in one readable line:

In [ ]:
medals.query("Country == 'USA'")[["Country", "Gold"]]

,Country,Gold
0,USA,46


> **&#9998; Exercise.** Select the countries that won fewer than 20 gold medals. Then select the countries that won more bronze medals than silver. You can compare two columns with each other just as you compared a column with a number.

In [ ]:
选出获得少于20枚金牌的国家，然后选择出获得铜牌多余银牌的国家。 可以将两列相互比较，就像一列与一个数字进行比较一样

In [ ]:
# your answer here
#import pandas as pd
#medals[medals["Gold"]<20]  # 找出金牌少于20的国家
#medals[medals["Bronze"]]>[medals["sliver"]]#铜牌 多余 银牌的国家
#medals[medals["Bronze"]>medals["sliver"]]
import pandas as pd
print(medals[medals["Gold"] < 20])            # 找出金牌少于20的国家
print(medals[medals["Bronze"] > medals["Silver"]])    # 铜牌多于银牌的国家

  Country  Gold  Silver  Bronze  Total
3     RUS    19      18      19     56
4     GER    17      10      15     42
5     JPN    12       8      21     41
  Country  Gold  Silver  Bronze  Total
0     USA    46      37      38    121
2     CHN    26      18      26     70
3     RUS    19      18      19     56
4     GER    17      10      15     42
5     JPN    12       8      21     41


### 3.5 Adding columns and rows

Because a column behaves like an array, you can do arithmetic on whole columns and store the result under a new name. This is how you add a calculated column:

In [ ]:
medals["Total"] = medals["Gold"] + medals["Silver"] + medals["Bronze"]
medals

,Country,Gold,Silver,Bronze,Total
0,USA,46,37,38,121
1,GBR,27,23,17,67
2,CHN,26,18,26,70
3,RUS,19,18,19,56
4,GER,17,10,15,42
5,JPN,12,8,21,41


No loop needed: pandas did all six rows at once. To remove a column again:

In [ ]:
medals = medals.drop(columns="Total")

Note the assignment. `drop()` hands back a new DataFrame and leaves the original alone, so without the `medals =` nothing would appear to happen. Many pandas methods work this way.

To add rows, build a small DataFrame with the same columns and glue the two together with `pd.concat()`:

In [ ]:
new_country = pd.DataFrame({"Country": ["FRA"],
                           "Gold": [10],
                           "Silver": [18], "Bronze": [14]})

medals = pd.concat([medals, new_country], ignore_index=True)
medals.tail(3)

,Country,Gold,Silver,Bronze
4,GER,17,10,15
5,JPN,12,8,21
6,FRA,10,18,14


The `ignore_index=True` part renumbers the rows 0, 1, 2 …. Without it the new row would keep its own index of 0 and you would end up with two rows both labelled 0, which causes confusion later.

### 3.6 Totals and sorting

Summing across the columns of each row is such a common job that there is a direct way to do it, using the same `axis` idea as the two-dimensional arrays in Part 2:

In [ ]:
medals["Total"] = medals[["Gold", "Silver", "Bronze"]].sum(axis=1)

Read `axis=1` as “collapse the columns, one number per row”. Leaving the axis out sums down each column instead, giving the column totals:

In [ ]:
medals[["Gold", "Silver", "Bronze", "Total"]].sum()

,0
Gold,157
Silver,132
Bronze,150
Total,439


Sorting is done with `sort_values()`:

In [ ]:
medals.sort_values("Bronze")     # smallest first, which is the default

,Country,Gold,Silver,Bronze,Total
6,FRA,10,18,14,42
4,GER,17,10,15,42
1,GBR,27,23,17,67
3,RUS,19,18,19,56
5,JPN,12,8,21,41
2,CHN,26,18,26,70
0,USA,46,37,38,121


In [ ]:
medals.sort_values("Bronze", ascending=False)     # largest first

,Country,Gold,Silver,Bronze,Total
0,USA,46,37,38,121
2,CHN,26,18,26,70
5,JPN,12,8,21,41
3,RUS,19,18,19,56
1,GBR,27,23,17,67
4,GER,17,10,15,42
6,FRA,10,18,14,42


In [ ]:
medals.sort_values("Total", ascending=False)

,Country,Gold,Silver,Bronze,Total
0,USA,46,37,38,121
2,CHN,26,18,26,70
1,GBR,27,23,17,67
3,RUS,19,18,19,56
4,GER,17,10,15,42
6,FRA,10,18,14,42
5,JPN,12,8,21,41


Notice that the index labels travel with their rows: row 0 is still the USA even though it is no longer at the top. Notice too that sorting hands back a sorted copy and leaves the original alone.

You can sort by one column and then another by passing a list of names, which is how you break ties. GER and FRA both have 42 medals, so a second column decides which comes first. Passing a list to ascending as well lets you mix the directions. Here that means most medals first, but fewest golds first among countries on the same total:

In [ ]:
medals.sort_values(["Total", "Gold"], ascending=[False, True])

,Country,Gold,Silver,Bronze,Total
0,USA,46,37,38,121
2,CHN,26,18,26,70
1,GBR,27,23,17,67
3,RUS,19,18,19,56
6,FRA,10,18,14,42
4,GER,17,10,15,42
5,JPN,12,8,21,41


### 3.7 Getting to know a new DataFrame

When a table lands on your desk for the first time, these are the things to reach for. Try each one on its own:

In [ ]:
medals.head()      # the first 5 rows - head(3) for the first 3
medals.tail()      # the last 5 rows
medals.shape      # (rows, columns)
len(medals)       # number of rows
medals.columns     # the column names
medals.info()     # column types, and how many values are missing
medals.describe()   # count, mean, std, min, quartiles, max per column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Country  7 non-null      object
 1   Gold     7 non-null      int64 
 2   Silver   7 non-null      int64 
 3   Bronze   7 non-null      int64 
 4   Total    7 non-null      int64 
dtypes: int64(4), object(1)
memory usage: 412.0+ bytes


,Gold,Silver,Bronze,Total
count,7.000000,7.000000,7.000000,7.000000
mean,22.428571,18.857143,21.428571,62.714286
std,12.204605,9.529403,8.343803,28.411936
min,10.000000,8.000000,14.000000,41.000000
25%,14.500000,14.000000,16.000000,42.000000
50%,19.000000,18.000000,19.000000,56.000000
75%,26.500000,20.500000,23.500000,68.500000
max,46.000000,37.000000,38.000000,121.000000


In [ ]:
medals.describe()

,Gold,Silver,Bronze,Total
count,7.000000,7.000000,7.000000,7.000000
mean,22.428571,18.857143,21.428571,62.714286
std,12.204605,9.529403,8.343803,28.411936
min,10.000000,8.000000,14.000000,41.000000
25%,14.500000,14.000000,16.000000,42.000000
50%,19.000000,18.000000,19.000000,56.000000
75%,26.500000,20.500000,23.500000,68.500000
max,46.000000,37.000000,38.000000,121.000000


`.describe()` is the quickest way to get a feel for a set of numbers, and it will be your first move on any new dataset. There is no separate viewer to open: put the name of a DataFrame on the last line of a cell and Colab displays it as a neat table. For a large table, look for the small table icon beside the output, which opens an interactive view you can sort and filter with the mouse.

### 3.8 Applying your own function to a column

Everything so far has used functions pandas gives you. When you want something of your own, `.apply()` runs a function on every value of a column. This is where the lambdas from Part 2 earn their keep, because the function is usually a one-liner:

In [ ]:
medals["Size"] = medals["Total"].apply(
    lambda t: "big" if t > 60 else "small")

medals[["Country", "Total", "Size"]]

,Country,Total,Size
0,USA,121,big
1,GBR,67,big
2,CHN,70,big
3,RUS,56,small
4,GER,42,small
5,JPN,41,small
6,FRA,42,small


A named function written with `def` works just as well, and is better as soon as the logic runs to more than one line. For straightforward arithmetic you do not need `.apply()` at all: column arithmetic like `medals["Gold"] * 2` is both simpler and faster.

> **&#9998; Exercise.** Use `.apply()` with a lambda to add a column holding each country's name in lower case. Then do the same with a named function and satisfy yourself that they agree.

In [ ]:
# your answer here
def to_lower(name):
  return name.lower(0)
  medals["lower"] = medals["country"].apply(to_lower)
  x = medals["lower"]
  print(x)


> **&#9998; Exercise.** Run `medals.describe()` and check that the mean of the Total column is what you would expect from the numbers you can see. Then use `.info()` and say how many rows and columns the table has.

In [ ]:
# your answer here
medals.describe()
medals.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Country  7 non-null      object
 1   Gold     7 non-null      int64 
 2   Silver   7 non-null      int64 
 3   Bronze   7 non-null      int64 
 4   Total    7 non-null      int64 
 5   Size     7 non-null      object
dtypes: int64(4), object(2)
memory usage: 468.0+ bytes


> **&#9998; Exercise.** Add a column called `GoldShare` holding the proportion of each country's medals that were gold, rounded to two decimal places. (Hint: divide two columns, then use `.round(2)`.) Which country has the highest share?

In [ ]:
# your answer here
medals["total"]=medals["Gold"]+medals["Silver"]+medals["Bronze"]
medals["goldShare"]=(medals["Gold"]/medals["total"]).round(2)
medals.sort_values("Gold")

,Country,Gold,Silver,Bronze,Total,Size,total,goldShare
6,FRA,10,18,14,42,small,42,0.24
5,JPN,12,8,21,41,small,41,0.29
4,GER,17,10,15,42,small,42,0.40
3,RUS,19,18,19,56,small,56,0.34
2,CHN,26,18,26,70,big,70,0.37
1,GBR,27,23,17,67,big,67,0.40
0,USA,46,37,38,121,big,121,0.38


## 4. Loading and saving data files

Typing a table by hand was fine for six countries. Real data comes in files, and the commonest format you will meet is CSV, comma separated values, which is plain text with one row per line and commas between the fields. Every spreadsheet program can produce one.

Because Colab runs on Google's computer rather than yours, the file has to get there before Python can read it. There are three ways.

1. **Upload it. **Open the Files pane from the left sidebar and click the upload button, then choose the file from your computer. This is the quickest option, but these files live only as long as your session does. When Colab disconnects you will need to upload again.
2. **Mount your Google Drive. **Click “Mount Drive” at the top of the Files pane, or run the two lines below. Colab asks your permission and then your whole Drive appears under /content/drive/MyDrive/. Files there are permanent, so this is the better option if you will come back to the same data each week.
3. **Read it straight from a web address. **pandas will take a URL instead of a file name.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Whichever route you take, the function that does the reading is `pd.read_csv()`. There is a copy of the full Rio 2016 medal table on Blackboard, called `Rio2016.csv`, with all the countries and their full names:

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")
rio.head()

#rio = pd.read_csv("Rio2016.csv")
#rio.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Country,Gold,Silver,Bronze
0,United States,46,37,38
1,South Korea,9,3,9
2,France,10,18,14
3,Australia,8,11,10
4,Germany,17,10,15


pandas takes the first line of the file as the column names, so there is nothing to configure for a normal CSV. Everything you have just learned then applies, because this is an ordinary DataFrame.

If the file will not load, check the name and spelling. `FileNotFoundError` almost always means the file is not where you told Python to look, and the Files pane shows you exactly what is there.

Writing a table back out is just as easy:

In [ ]:
rio.to_csv("my_medals.csv", index=False)

The `index=False` stops the row labels being written out as an extra unnamed first column, which is almost always what you want. Try it both ways and look at the two files. You can open a CSV in the Files pane, or in Excel.

That file is on Google's machine and will disappear when the session ends. To keep it, either download it to your own computer:

In [ ]:
from google.colab import files
files.download("my_medals.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

or save it into your Google Drive, which is permanent:

In [ ]:
rio.to_csv("/content/drive/MyDrive/my_medals.csv", index=False)

> **&#9998; Exercise.** Load `Rio2016.csv`, add a Total column, and sort the table by the total number of medals won, largest first. Which countries make the top five?

In [ ]:
# your answer here
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")

rio.head()
rio["Total"] = rio["Gold"] + rio["Silver"] + rio["Bronze"]
rio.sort_values("Total", ascending=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Country,Gold,Silver,Bronze,Total
0,United States,46,37,38,121
12,China,26,18,26,70
8,United Kingdom,27,23,17,67
6,Russia,19,18,19,56
2,France,10,18,14,42
...,...,...,...,...,...
73,Austria,0,0,1,1
71,Estonia,0,0,1,1
80,Jordan,1,0,0,1
84,Puerto Rico,1,0,0,1


> **&#9998; Exercise.** Using the same file, calculate the total number of gold, silver and bronze medals won at the Games by all countries together. Then save your sorted table to a new CSV file and find it in the Files pane.

In [ ]:
# your answer here
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")           #调用google drive的 rio 2016

rio["Total"] = rio["Gold"] + rio["Silver"] + rio["Bronze"]        # 先加 Total 列
rio_sorted = rio.sort_values("Total", ascending=False)           # 一个等号，大写的 T

print(rio[["Gold", "Silver", "Bronze"]].sum())               # 所有国家的奖牌总数

rio_sorted.to_csv("/content/drive/MyDrive/rio_sorted.csv", index=False)   # 保存
rio_sorted.head()                              # 放在最后一行才会显示出来




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Gold      307
Silver    307
Bronze    360
dtype: int64


,Country,Gold,Silver,Bronze,Total
0,United States,46,37,38,121
12,China,26,18,26,70
8,United Kingdom,27,23,17,67
6,Russia,19,18,19,56
2,France,10,18,14,42


## 5. Cleaning messy data

Rio2016.csv is a tidy file. Most files are not. Data that has been typed by hand, exported from a spreadsheet or scraped off a website arrives with blanks, duplicates, stray spaces and numbers that Python thinks are text. Finding and fixing those faults is most of the work in any real project, and it is often the difference between an answer and a wrong answer.

Here is a small table with a deliberate fault of each kind, short enough that you can see every problem on screen. Real data never is.

In [ ]:
messy = pd.DataFrame({
    "Country": [" USA", "gbr", "CHN", "GBR ", "JPN", "chn "],
    "Gold":    ["46", "27", "26", "27", "12", "26"],
    "Silver":  [37, 23, 18, 23, 8, 18],
    "Bronze":  [38, 17, None, 17, 21, None],
})
messy

,Country,Gold,Silver,Bronze
0,USA,46,37,38.0
1,gbr,27,23,17.0
2,CHN,26,18,NaN
3,GBR,27,23,17.0
4,JPN,12,8,21.0
5,chn,26,18,NaN


### 5.1 Look before you leap

Your first move on any new table is `.info()`. It reports the type of every column and how many values each one actually holds.

In [ ]:
messy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Country  6 non-null      object 
 1   Gold     6 non-null      object 
 2   Silver   6 non-null      int64  
 3   Bronze   4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 324.0+ bytes


Two things in that output matter. The Dtype column says Gold is text, `object` or `str` depending on your version of pandas, rather than a number; the values were typed with quotation marks round them. And the Non-Null Count says Bronze holds four values where every other column has six, so two are missing.

To count the missing values directly, `.isna()` marks every cell True or False, and `.sum()` adds up the Trues column by column:

In [ ]:
print(messy.isna().sum())

Country    0
Gold       0
Silver     0
Bronze     2
dtype: int64


### 5.2 Tidying text

Look at the country names. Two carry stray spaces and two are in lower case, so Python sees six different countries where there are only four. String methods reach a whole column at once through .str:

In [ ]:
messy["Country"] = messy["Country"].str.strip().str.upper()
messy

,Country,Gold,Silver,Bronze
0,USA,46,37,38.0
1,GBR,27,23,17.0
2,CHN,26,18,NaN
3,GBR,27,23,17.0
4,JPN,12,8,21.0
5,CHN,26,18,NaN


`.str.strip()` removes spaces from both ends, and `.str.upper()` puts everything in capitals. `.str.lower()` and `.str.title()` do the obvious things, and `.str.replace("old", "new")` swaps one piece of text for another.

### 5.3 Removing duplicate rows

Now that the names match, the repeats are visible: GBR and CHN each appear twice. The order of those two steps matters. Had we looked for duplicates first, Python would have found none, because “gbr” and “GBR ” are different pieces of text. Clean the text, then deduplicate.

In [ ]:
print(messy.duplicated().sum())

2


In [ ]:
clean = messy.drop_duplicates().reset_index(drop=True)
clean

,Country,Gold,Silver,Bronze
0,USA,46,37,38.0
1,GBR,27,23,17.0
2,CHN,26,18,NaN
3,JPN,12,8,21.0


`drop_duplicates()` keeps the first copy of each repeated row and discards the rest. `.reset_index(drop=True)` then renumbers the rows, because the discarded ones leave gaps in the index.

### 5.4 Numbers stored as text

The Gold column still holds text. That is worse than it sounds, because arithmetic on text does do something; it just does the wrong thing.

In [ ]:
print(clean["Gold"].sum())

46272612


Python has glued the four values end to end rather than adding them up, exactly as it would for any other strings. Nothing raised an error, which is precisely why this one catches people out.

In [ ]:
clean["Gold"] = clean["Gold"].astype(int)
print(clean["Gold"].sum())

111


`.astype(int)` converts a whole column, and `.astype(float)` does the same for decimals. It refuses if any value cannot be converted, which is useful: the error message names the value that is the problem.

### 5.5 Missing values

Two blanks are left in Bronze, shown as NaN, which stands for “not a number”. You have two choices, and which is right depends entirely on what the blank means.

* `.dropna()` throws away any row with a missing value. Sensible when the row is useless without it.
* `.fillna(0)` puts a value into the gap. Sensible only if you are confident the blank really does mean zero.

A missing medal count plausibly means none were won, so filling with zero is defensible here. On a column of temperatures or salaries it would not be, and quietly inventing zeros would wreck every average you then calculated.

In [ ]:
clean["Bronze"] = clean["Bronze"].fillna(0).astype(int)
clean

,Country,Gold,Silver,Bronze
0,USA,46,37,38
1,GBR,27,23,17
2,CHN,26,18,0
3,JPN,12,8,21


> **Note:** a column holding any missing value is stored as decimals, which is why Bronze read 38.0 rather than 38. Once the gaps are filled you can convert it back to whole numbers, which is what the .astype(int) above is doing.

### 5.6 Column names

Names imported from a spreadsheet often carry capitals, trailing spaces and the odd line break, which makes them tiresome to type. The same string methods work on them:

In [ ]:
print(list(clean.columns.str.strip().str.lower()))

['country', 'gold', 'silver', 'bronze']


To make that stick, assign it back with `clean.columns = ...`. To change one or two names rather than all of them, use `.rename(columns={"old": "new"})`, which hands back a new DataFrame.

> **&#9998; Exercise.** Load `Rio2016.csv` again and run `.info()` on it. Does it have any missing values? Is every column that ought to be a number actually stored as one?

In [ ]:
# your answer here
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")

rio.info()
print(rio.isna().sum())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87 entries, 0 to 86
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Country  87 non-null     object
 1   Gold     87 non-null     int64 
 2   Silver   87 non-null     int64 
 3   Bronze   87 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 2.8+ KB
Country    0
Gold       0
Silver     0
Bronze     0
dtype: int64


> **&#9998; Exercise.** Check the country names in `Rio2016.csv` for stray spaces, by comparing the column with a stripped copy of itself and counting how many differ. Compare two columns with `!=` and count the Trues with `.sum()`.

In [ ]:
# your answer here
#import pandas as pd
#from google.colab import drive
#drive.mount("/content/drive")

#rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")
stripped = rio["Country"].str.strip()

diff = rio["Country"]!=stripped
print(diff.sum())


0


> **&#9998; Exercise.** Add a row to `clean` that duplicates one already there, using `pd.concat()` as you did earlier. Then run the duplicate check and the drop again, and satisfy yourself that it is caught.

In [ ]:
给 `clean` 这个 DataFrame 新增一行，这一行要和表里已有的某一行完全重复。

In [ ]:
# your answer here
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

rio = pd.read_csv("/content/drive/MyDrive/Rio2016.csv")

dup_row = clean.iloc[[1]]
clean_test = pd.concat([clean,dup_row],ignore_index=True)
clean_test

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Country,Gold,Silver,Bronze
0,USA,46,37,38
1,GBR,27,23,17
2,CHN,26,18,0
3,JPN,12,8,21
4,GBR,27,23,17


## 6. Final exercise

This one brings the whole session together: load a real file, get it into shape, answer a question with it, and save the result. Work in a fresh set of cells at the bottom of your notebook.

1. Load `Rio2016.csv` into a DataFrame and get a first look at it with `.head()`, `.shape` and `.info()`.
2. Check it for the faults from the last section: missing values, duplicate rows, and columns that ought to be numbers but are not. Fix anything you find, and say in a text cell what you fixed, or why nothing needed fixing.
3. Add a Total column holding each country's medals added together.
4. Keep only the countries that won at least ten medals in total, and sort what is left with the most successful first.
5. Add a column giving each country's golds as a share of its total, rounded to two decimal places. Use `.apply()` with a lambda, or plain column arithmetic, whichever you prefer.
6. Save your finished table into your Google Drive as a CSV, and check it is there.

Finish with a short text cell, three or four sentences, saying what your table shows: which countries dominate, whether a large medal haul always comes with a high gold share, and one thing that surprised you. A table nobody can interpret is not finished.

Save the notebook to your Drive when you are done. You will want it again next week.

Useful resources

* 10 minutes to pandas: [https://pandas.pydata.org/docs/user_guide/10min.html](https://pandas.pydata.org/docs/user_guide/10min.html)
* pandas cheat sheet: [https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
* Working with missing data, the pandas user guide: [https://pandas.pydata.org/docs/user_guide/missing_data.html](https://pandas.pydata.org/docs/user_guide/missing_data.html)
* A very large collection of practice datasets, each with a CSV link you can hand straight to pd.read_csv(): [https://vincentarelbundock.github.io/Rdatasets/datasets.html](https://vincentarelbundock.github.io/Rdatasets/datasets.html)
* Python for Data Analysis by Wes McKinney, Chapters 5, 6 and 7: [https://wesmckinney.com/book/](https://wesmckinney.com/book/)